In [ ]:
# === Notebook common preamble (load the llm_math package) ===
import sys
from pathlib import Path

# Candidate paths for the llm_math package
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# Add parent directories as candidates (when running from the notebooks folder)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# Try importing llm_math
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[Warning] load the llm_math package text: {e}")
    print("  Clone the GitHub repository and run colab_setup.sh.")
# === end preamble ===


# Ch 23. KV Cache — Inference Speedtext text [CPU/GPU Benchmark ⑩]

> **Learning Goals**
> - KV Cachetext text Inference Speedtext $O(n^2) \to O(n)$text text text
> - Prefill vs Decode text text
> - text text Calculationtext text
> - KV Cache text text/text text Speedtext Comparisontext

## 23.1 Inference text: Prefill vs Decode

LLM Inferencetext text text:
1. **Prefill (Context Encoding)**: text text text. text text. text $O(n^2)$.
2. **Decode (Token Generation)**: text text text. text. text text $O(n)$.

Decodetext text Timetext text. KV Cachetext decode Speedtext text text.

## 23.2 text K, Vtext text

Attention: $\mathrm{softmax}(QK^\top/\sqrt{d_k})V$

text $t$ text text:
- $Q_t$ (text text) text
- $K_{\leq t}, V_{\leq t}$ (text text text, Value) text

text text text text text text text $K, V$text textCalculation → $O(t^2)$.
text text $K_{\leq t-1}, V_{\leq t-1}$ text $K_t, V_t$text text → $O(t)$.

**Qtext text text text**: text text text $Q_t$ text text.


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import time

# text text Attention (text text text textCalculation)
def attention_no_cache(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-1, -2) / (d_k ** 0.5)
    if mask is not None:
        scores = scores + mask
    weights = F.softmax(scores, dim=-1)
    return weights @ V

# text text Attention (text)
def attention_with_cache(q_new, k_cache, v_cache, k_new, v_new):
    """q_new: (B, 1, d), k_cache/v_cache: (B, T, d)"""
    # Cachetext text k, v text
    k_full = torch.cat([k_cache, k_new], dim=1)  # (B, T+1, d)
    v_full = torch.cat([v_cache, v_new], dim=1)
    d_k = q_new.shape[-1]
    scores = q_new @ k_full.transpose(-1, -2) / (d_k ** 0.5)  # (B, 1, T+1)
    weights = F.softmax(scores, dim=-1)
    out = weights @ v_full  # (B, 1, d)
    return out, k_full, v_full

# Comparison: 100 text Generation
np.random.seed(0)
torch.manual_seed(0)
B, d, T = 1, 64, 1
n_generate = 50

# Cache text text
t0 = time.perf_counter()
Q = torch.randn(B, T, d)
K = torch.randn(B, T, d)
V = torch.randn(B, T, d)
for step in range(n_generate):
    # text Step text textComputation
    Q_new = torch.randn(B, 1, d)
    Q = torch.cat([Q, Q_new], dim=1)
    K = torch.cat([K, torch.randn(B, 1, d)], dim=1)
    V = torch.cat([V, torch.randn(B, 1, d)], dim=1)
    out = attention_no_cache(Q, K, V)
t_no_cache = time.perf_counter() - t0
print(f"text text: {t_no_cache*1000:.2f}ms ({n_generate} text)")

# Cache text text
torch.manual_seed(0)
t0 = time.perf_counter()
k_cache = torch.randn(B, 1, d)
v_cache = torch.randn(B, 1, d)
for step in range(n_generate):
    q_new = torch.randn(B, 1, d)
    k_new = torch.randn(B, 1, d)
    v_new = torch.randn(B, 1, d)
    out, k_cache, v_cache = attention_with_cache(q_new, k_cache, v_cache, k_new, v_new)
t_cache = time.perf_counter() - t0
print(f"text text: {t_cache*1000:.2f}ms ({n_generate} text)")
print(f"Speed Improvement: {t_no_cache/t_cache:.2f}x")


## 23.3 KV Cache text Calculation

KV Cache text = $2 \cdot L \cdot h \cdot d_k \cdot n \cdot \text{dtype}$

- $L$: text text
- $h$: KV text text (MHAtext $h$, MQAtext 1, GQAtext $g$)
- $d_k$: text Dimension
- $n$: Sequence Length
- 2: Ktext V
- dtype: FP16=2 bytes, FP32=4 bytes

text: LLaMA-7B ($L=32, h=32, d_k=128$), FP16, $n=4096$:
$$2 \times 32 \times 32 \times 128 \times 4096 \times 2 = 2 \text{ GB}$$

$n=32768$text 16GB → GPU text text text.


In [ ]:
# KV Cache text Calculationtext
def kv_cache_memory_gb(n_layers, n_kv_heads, d_k, seq_len, dtype_bytes=2):
    """KV Cache Memory (GB)."""
    return 2 * n_layers * n_kv_heads * d_k * seq_len * dtype_bytes / (1024**3)

# text Model/Sequence Length
models = [
    ('LLaMA-7B', 32, 32, 128, 2),
    ('LLaMA-13B', 40, 40, 128, 2),
    ('LLaMA-70B', 80, 8, 128, 2),  # GQA
    ('GPT-2 small', 12, 12, 64, 2),
    ('Mistral-7B', 32, 8, 128, 2),  # GQA
]
seq_lens = [2048, 8192, 32768, 131072]

print(f"{'Model':>15} | ", end='')
for sl in seq_lens:
    print(f"n={sl:>6}", end=' | ')
print()
print("-" * 75)
for name, L, h, dk, dt in models:
    print(f"{name:>15} | ", end='')
    for sl in seq_lens:
        mem = kv_cache_memory_gb(L, h, dk, sl, dt)
        print(f"{mem:>7.2f}GB", end=' | ')
    print()

print("\n=> Sequence Lengthtext textPlane KV Cachetext Memory textSubspacetext text.")
print("=> GQA(Mistral, LLaMA-70B)text KV Head text text Memory Efficiencytext.")


## 23.4 [CPU/GPU Benchmark ⑩] KV Cache on/off


In [ ]:
# KV Cache Benchmark
from llm_math.bench import time_fn

# text Attention text text text text
def bench_decode_no_cache(n_context, d=64, n_decode=20):
    """Cache text n_decode text Generation."""
    Q = torch.randn(1, n_context, d)
    K = torch.randn(1, n_context, d)
    V = torch.randn(1, n_context, d)
    for _ in range(n_decode):
        Q_new = torch.randn(1, 1, d)
        Q = torch.cat([Q, Q_new], dim=1)
        K = torch.cat([K, torch.randn(1, 1, d)], dim=1)
        V = torch.cat([V, torch.randn(1, 1, d)], dim=1)
        _ = Q @ K.transpose(-1, -2) / (d ** 0.5) @ V

def bench_decode_with_cache(n_context, d=64, n_decode=20):
    """Cache text n_decode text Generation."""
    k_cache = torch.randn(1, n_context, d)
    v_cache = torch.randn(1, n_context, d)
    for _ in range(n_decode):
        q_new = torch.randn(1, 1, d)
        k_new = torch.randn(1, 1, d)
        v_new = torch.randn(1, 1, d)
        k_full = torch.cat([k_cache, k_new], dim=1)
        v_full = torch.cat([v_cache, v_new], dim=1)
        _ = q_new @ k_full.transpose(-1, -2) / (d ** 0.5) @ v_full
        k_cache = k_full
        v_cache = v_full

print(f"{'n_context':>10} | {'No Cache (ms)':>15} | {'Cache (ms)':>12} | {'Speedup':>10}")
print("-" * 55)
for n_ctx in [128, 512, 1024, 2048]:
    t_no = time_fn(bench_decode_no_cache, n_ctx, device='cpu', warmup=1, repeat=3)['mean_ms']
    t_yes = time_fn(bench_decode_with_cache, n_ctx, device='cpu', warmup=1, repeat=3)['mean_ms']
    print(f"{n_ctx:>10} | {t_no:>15.3f} | {t_yes:>12.3f} | {t_no/t_yes:>9.2f}x")
print("\n=> Contexttext text KV Cachetext Effecttext text (O(n) vs O(n^2)).")


## 23.5 Continuous Batching

text batching: text text text text text text text → text text text text text.

**Continuous Batching** (vLLM text): text text text text text text text.
- text(thoughtput) text text
- text text text text text

## 23.6 PagedAttention (vLLM)

OS text text:
- KV Cachetext text text "text"text text
- text text text
- text text text

text text text, text text text text. vLLMtext 2-4x text text.

## 23.7 Key Takeaways

| text | text |
|---|---|
| Prefill | text text text, $O(n^2)$ |
| Decode | text text, $O(n)$ per token (KV cache) |
| KV Cache | $K, V$ text, text text $K_t, V_t$text text |
| text | $2 L h d_k n \cdot \text{dtype}$ |
| Continuous Batching | text text, text text |
| PagedAttention | text text text |

## Exercises

1. KV Cache on/offtext 100text text Timetext n=128, 512, 2048text Comparisontext.
2. LLaMA-7Btext text 32Ktext KV Cache text Calculationtext.
3. GQAtext KV Cache text text text LLaMA-7B(32 text) vs LLaMA-70B(8 KV text)text Comparisontext.
4. Continuous Batchingtext text text text text.
5. PagedAttentiontext text text text text text.

> Solutions: `solutions/ch23_solutions.ipynb`
